# Elizabeth: weak and strong formulations

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/optimization/elizabeth-location-models.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/foundations/optimization/elizabeth-location-models.ipynb)

**Worked Example** · UvA Business Analytics teaching collection

Work through the questions before running each cell. Explain the result, check its assumptions, and change one input to test your understanding.


## Setup
Use the installed scientific libraries and install only missing dependencies. The shared helper keeps this routine code in one place. Solver-specific lessons introduce additional solvers at the point where they are used.


In [ ]:
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'highspy': 'highspy', 'numpy': 'numpy', 'pyomo': 'pyomo'}
ensure_packages(required_packages)


In [ ]:
import pyomo.environ as pyo
from pyomo.opt import assert_optimal_termination
SOLVER = "appsi_highs"
assert pyo.SolverFactory(SOLVER).available(exception_flag=False)


## The same integer decisions, different relaxations
Let $y_j$ indicate whether facility $j$ opens and let $x_{ij}$ assign customer $i$ to facility $j$. Each customer must be assigned once. We compare
$$\sum_i x_{ij}\leq |I|y_j \quad\hbox{and}\quad x_{ij}\leq y_j\;\;\forall i,j.$$
Both are correct for binary variables. The second family implies the first when summed over customers and gives a tighter LP relaxation. This says something about the bound, not a guarantee about solver running time on every instance.


In [ ]:
import numpy as np
rng = np.random.default_rng(42)
facilities = rng.uniform(0,100,(6,2))
customers = rng.uniform(0,100,(24,2))
installation = rng.uniform(100,200,6)
service = np.linalg.norm(customers[:,None,:]-facilities[None,:,:],axis=2)


In [ ]:
def location_model(strong=True, relax=False):
    m = pyo.ConcreteModel('Elizabeth')
    m.I = pyo.RangeSet(0,len(customers)-1)
    m.J = pyo.RangeSet(0,len(facilities)-1)
    domain = pyo.UnitInterval if relax else pyo.Binary
    m.open = pyo.Var(m.J,domain=domain)
    m.assign = pyo.Var(m.I,m.J,domain=domain)
    m.cost = pyo.Objective(expr=pyo.quicksum(float(installation[j])*m.open[j] for j in m.J)
        + pyo.quicksum(float(service[i,j])*m.assign[i,j] for i in m.I for j in m.J))
    @m.Constraint(m.I)
    def once(m,i):
        return pyo.quicksum(m.assign[i,j] for j in m.J) == 1
    if strong:
        @m.Constraint(m.I,m.J)
        def link(m,i,j):
            return m.assign[i,j] <= m.open[j]
    else:
        @m.Constraint(m.J)
        def link(m,j):
            return pyo.quicksum(m.assign[i,j] for i in m.I) <= len(customers)*m.open[j]
    return m

values = {}
for strong in [False,True]:
    for relax in [False,True]:
        model = location_model(strong,relax)
        result = pyo.SolverFactory(SOLVER).solve(model)
        assert_optimal_termination(result)
        values[strong,relax] = pyo.value(model.cost)
print(values)
assert abs(values[False,False]-values[True,False]) < 1e-5
assert values[False,True] <= values[True,True] + 1e-5
assert values[True,True] <= values[True,False] + 1e-5


## Interpret the comparison
Explain why the inequality between the LP bounds has this direction for a minimization model. Change the installation costs and repeat. Do not confuse a stronger formulation with a different underlying facility-location problem.


## Install the commercial solver interfaces at this stage
The original lesson adds Gurobi, CPLEX and Xpress here, after the open-source comparison. We retain that sequence. Install only missing packages, then actually solve both formulations using each solver. The small instance above fits the packaged small-model editions.

The packaged small-model editions have documented limits: [Gurobi](https://support.gurobi.com/hc/en-us/articles/29682074018833-What-does-Restricted-license-for-non-production-use-only-mean), [CPLEX](https://www.ibm.com/products/ilog-cplex-optimization-studio/pricing), and [Xpress](https://github.com/fico-xpress/xpress-training/blob/main/python/md/python-full-course.md). If installation or licensing fails, resolve the error; a skipped solver is not a completed comparison.


In [ ]:
from teaching_utils import ensure_packages, solve_checked
required_packages = {'gurobipy': 'gurobipy', 'cplex': 'cplex', 'xpress': 'xpress', 'pandas': 'pandas'}
ensure_packages(required_packages)


In [ ]:
from time import perf_counter
import pandas as pd
comparison_rows = []
for name in ('appsi_highs', 'gurobi_direct', 'cplex_direct', 'xpress_direct'):
    for strong in (False, True):
        model = location_model(strong=strong)
        started = perf_counter()
        result = solve_checked(model, name)
        comparison_rows.append({'solver': name, 'strong': strong,
                                'objective': pyo.value(model.cost),
                                'seconds': perf_counter() - started})
comparison = pd.DataFrame(comparison_rows)
assert comparison['solver'].nunique() == 4
assert comparison['objective'].max() - comparison['objective'].min() < 1e-5
display(comparison)
